**General Description**

The following notebook contains the code to create, train, validate, and test a rainfall-runoff model using an LSTM network architecture, using the Caravan dataset. The details for the experiment can be read from a .yml file.

***Authors:***
- Sanika Baste (sanika.baste@kit.edu)

In [1]:
import datetime
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

from hy2dl.datasetzoo import get_dataset
from hy2dl.evaluation import calculate_metrics, get_tester
from hy2dl.modelzoo import get_model
from hy2dl.training.basetrainer import BaseTrainer
from hy2dl.utils.config import Config

base_dir = Path.cwd().resolve()
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}

/home/ka/ka_iwu/ka_si3685/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Part 1. Initialize information

In [2]:
# Path to .yml file where the experiment settings are stored.
path_experiment_settings = "../examples/configs/caravan.yml"

# Read experiment settings
config = Config(path_experiment_settings, base_dir=base_dir)
config.init_experiment()
config.dump()

Dataset = get_dataset(config)
Tester = get_tester(config)

Part 2. Create datasets and dataloaders used to train/validate the model

In [3]:
# Create training dataset
training_dataset = Dataset(cfg=config, time_period="training")
training_dataset.setup_dataset()
# Initialize training object
trainer = BaseTrainer(cfg=config, training_dataset=training_dataset)

2026-07-27 10:11:23 - Creating training dataset in memory...


Processing gauges: 100%|##########| 17847/17847 [07:12<00:00, 41.30entity/s]
2026-07-27 10:18:46,470 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-07-27 10:18:46,470 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-07-27 10:18:46,471 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-07-27 10:18:46,471 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing


2026-07-27 10:18:59 - Dataset created successfully.
2026-07-27 10:19:00 - Validating samples...


Validating samples: 100%|##########| 517564/517564 [01:48<00:00, 4754.88tasks/s]


2026-07-27 10:23:29 - Number of gauges with valid samples: 17474
2026-07-27 10:23:29 - Gauges without valid samples in period of interest: GRDC_1160095, GRDC_1160774, GRDC_1160777, GRDC_1196105, GRDC_1321880, GRDC_1322880, GRDC_3265070, GRDC_3265100, GRDC_3275270, GRDC_3275910, GRDC_4101300, GRDC_4105060, GRDC_4213684, GRDC_5101162, GRDC_5109201, GRDC_5204102, GRDC_6172015, GRDC_6172032, GRDC_6220345, GRDC_6227440, GRDC_6228820, camelsch_2620, camelsch_2630, camelsch_2631, camelsch_4019, camelscl_10122003, camelscl_10322003, camelscl_10343002, camelscl_10344004, camelscl_12400004, camelscl_12561001, camelscl_12820001, camelscl_12825002, camelscl_1300009, camelscl_6000003, camelscl_6034022, camelscl_8140002, camelscl_8821006, camelscz_H4014800, camelscz_H4055500, camelscz_H4057200, camelscz_L4186900, camelscz_O4360900, camelscz_O4370500, camelscz_P4098000, camelscz_P4163200, camelscz_P4165800, camelscz_U4222900, camelsde_DE110890, camelsde_DE112400, camelsde_DE112420, camelsde_DE112450,

In [4]:
validation_dataset = Dataset(cfg=config, time_period="validation")
validation_dataset.setup_dataset(check_nan=False, path_scaler = config.path_save_folder / "scaler.yml" )
tester_validation = Tester(cfg=config, evaluation_dataset=validation_dataset)

2026-07-27 10:32:44 - Creating validation dataset in memory...


Processing gauges: 100%|##########| 17847/17847 [07:16<00:00, 40.90entity/s]
2026-07-27 10:40:13,009 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-07-27 10:40:13,010 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-07-27 10:40:13,010 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-07-27 10:40:13,011 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing


2026-07-27 10:40:13 - Dataset created successfully.
2026-07-27 10:40:15 - Validating samples...


Validating samples: 100%|##########| 142777/142777 [00:12<00:00, 11232.98tasks/s]


2026-07-27 10:40:38 - Number of gauges with valid samples: 17847
2026-07-27 10:41:28 - Number of valid samples: 32 588 622
2026-07-27 10:41:28 - Mapping ids, dates and features to their corresponding indexes
2026-07-27 10:41:53 - Dataset was successfully standardized.
2026-07-27 10:41:53 - Static attributes were successfully standardized.
2026-07-27 10:42:05 - Time required to process the dataset: 0:09:21


Part 3. Train model

In [5]:
# Training report structure
validation_headers = "".join([f"{m:^10}|" for m in config.validation_metric])
config.logger.info("Training model".center(60, "-"))
config.logger.info(f"{'':^16}|{'Training':^21}|{'Validation':^{(11 * len(config.validation_metric)) + 10}}|")
config.logger.info(f"{'Epoch':^5}|{'LR':^10}|{'Loss':^10}|{'Time':^10}|{validation_headers}{'Time':^10}|")

# Loop through epochs
total_time = time.time()
for epoch in range(1, config.epochs + 1):
    trainer.train_model(epoch=epoch)  # Training
    tester_validation.validate_model(model=trainer.model, epoch=epoch)  # Validation
    config.logger.info(trainer.report + tester_validation.validation_report)  # report

config.logger.info(f"Total training time: {datetime.timedelta(seconds=int(time.time() - total_time))}\n")
shutil.rmtree(tester_validation.path_zarr, ignore_errors=True)  # delete validation results

2026-07-27 10:42:19 - -----------------------Training model-----------------------
2026-07-27 10:42:19 -                 |      Training       |     Validation      |
2026-07-27 10:42:19 - Epoch|    LR    |   Loss   |   Time   |   nse    |   Time   |


KeyboardInterrupt: 

Part 4. Test model

In [ ]:
# If I already trained a model, I can re-construct it using the saved parameters from a given epoch
#model = get_model(config).to(config.device)
#model.load_state_dict(torch.load(config.path_save_folder / "model" / f"model_epoch_{config.epochs}", map_location=config.device))

In [ ]:
testing_dataset = Dataset(cfg=config, time_period="testing")
testing_dataset.setup_dataset(check_nan=False, path_scaler = config.path_save_folder / "scaler.yml" )
tester_testing = Tester(cfg=config, evaluation_dataset=testing_dataset)

config.logger.info("Testing model...")
testing_time = time.time()
tester_testing.evaluate_model(model = trainer.model)
config.logger.info("Testing completed.")
config.logger.info(f"Total testing time: {datetime.timedelta(seconds=int(time.time() - testing_time))}\n")

Part 5. Initial analysis

In [ ]:
test_results = xr.open_zarr(tester_testing.path_zarr)
testing_metrics = calculate_metrics(ds_results=test_results, metric_name = config.testing_metrics)
testing_metrics.to_zarr(config.path_save_folder / "testing_metrics.zarr", mode="w")

In [ ]:
# Loss testing
target_of_interest = random.sample(list(testing_metrics.feature.values), 1)[0]
test_metric = testing_metrics.sel(feature=target_of_interest, metric="nse").round(3).T.to_pandas().dropna()
# Plot the histogram
plt.figure(figsize=(10, 5))
plt.hist(test_metric, bins = np.linspace(0.0, 1.0, 11).tolist())
# Add NSE statistics to the plot
plt.text(
    0.01,
    0.8,
    (
        f"Mean: {'%.2f' % test_metric.mean():>7}\n"
        f"Median: {'%.2f' % test_metric.median():>0}\n"
        f"Max: {'%.2f' % test_metric.max():>9}\n"
        f"Min: {'%.2f' % test_metric.min():>10}"
    ),
    transform=plt.gca().transAxes,
    bbox=dict(facecolor="white", alpha=0.5),
)

# Format plot
plt.xlabel("NSE", fontsize=12, fontweight="bold")
plt.ylabel("Frequency", fontsize=12, fontweight="bold")
plt.title(f"NSE histogram for: {target_of_interest}", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Plot simulated and observed discharges
basin_to_analyze = random.sample(list(test_results.gauge_id.values), 1)[0]
y_sim = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_sim"].compute().values
y_obs = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_obs"].compute().values

plt.figure(figsize=(15, 7.5))
plt.plot(y_obs, label="observed", color=color_palette["observed"])
plt.plot(y_sim, label="simulated", alpha=0.5, color=color_palette["simulated"])

# Format plot
plt.xlabel("Date", fontsize=12, fontweight="bold")
plt.ylabel(target_of_interest, fontsize=12, fontweight="bold")
plt.title(f"Results for gauge_id: {basin_to_analyze}", fontsize=16, fontweight="bold")
plt.tick_params(axis="both", which="major", labelsize=12)
plt.legend(loc="upper right", fontsize=12)
plt.tight_layout()
plt.show()